In [16]:
# ==============================================================================
# BƯỚC 0: IMPORT CÁC THƯ VIỆN CẦN THIẾT
# ==============================================================================
import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd
from collections import Counter

from underthesea import sent_tokenize as vi_sent_tokenize
from underthesea import word_tokenize as vi_word_tokenize

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', None)

print("-> Đã import thư viện thành công!")


-> Đã import thư viện thành công!


In [ ]:
# ==============================================================================
# BƯỚC 1: CRAWL TOÀN BỘ VĂN BẢN (TEXT EXTRACTION) TỪ VNEXPRESS
# ==============================================================================
url = "https://vnexpress.net/ket-qua-viet-nam-vs-pakistan-5127670.html"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
response.encoding = "utf-8"

soup = BeautifulSoup(response.text, "html.parser")


title_tag = soup.find("h1", class_="title-detail")
title = title_tag.get_text(strip=True) if title_tag else ""

desc_tag = soup.find("p", class_="description")
description = desc_tag.get_text(strip=True) if desc_tag else ""

paragraphs = [p.get_text(strip=True) for p in soup.find_all("p", class_="Normal") if p.get_text(strip=True)]

all_parts = [title, description] + paragraphs
full_text = unicodedata.normalize("NFC", "\n".join(all_parts))

print(f"=== TOÀN BỘ VĂN BẢN ĐÃ CRAWL ({len(paragraphs)} đoạn văn, {len(full_text)} ký tự) ===\n")
print(full_text)


=== TOÀN BỘ VĂN BẢN ĐÃ CRAWL (20 đoạn văn, 4010 ký tự) ===

Việt Nam thắng 4-2, xếp nhì bảng FIFA ASEAN Cup
IndonesiaTiền đạo Nguyễn Đình Bắc ghi một bàn, kiến tạo hai bàn sau khi vào sân từ ghế dự bị, giúp Việt Nam thắng Pakistan 4-2 để giành suất tranh giải ba FIFA ASEAN Cup 2026.
Việt Nam thắng 4-2, qua đó giành sáu điểm qua ba trận, xếp nhì bảng B. Thầy trò Kim Sang-sik sẽ đấu Malaysia ở trận tranh giải ba FIFA ASEAN Cup hôm 5/10.
Tiền đạo 19 tuổi ghi bàn đầu tiên cho đội tuyển. Từ đường chuyền của Đình Bắc, Minh Hoàng thoát xuống đánh bại thủ môn Pakistan trong pha đối mặt, nâng tỷ số lên 4-2.
Hiệp hai có 4 phút bù giờ.
Ali Khan dứt điểm, bóng bất ngờ chạm chân Kaleem Ulah đổi hướng vào lưới. Patrik Lê Giang đã đổ người trước đó, nên không kịp phản ứng.
Phan Tuấn Tài chuyền vượt tuyến để Đình Bắc thoát xuống sau lưng hàng thủ Pakistan. Tiền đạo Việt Nam bình tĩnh dứt điểm trong thế đối mặt thủ môn, nâng cách biệt lên hai bàn.
Hai Long bứt tốc đón đường chuyền dài của Xuân Mạnh rồi

In [22]:
# ==============================================================================
# BƯỚC 2: TÁCH CÂU (SENTENCE SEGMENTATION)
# ==============================================================================
sentences = vi_sent_tokenize(full_text)

print(f"=== TÁCH CÂU THÀNH CÔNG (TỔNG CỘNG: {len(sentences)} CÂU) ===\n")
df_sentences = pd.DataFrame({
    "STT": list(range(1, len(sentences) + 1)),
    "Nội dung câu": sentences
})
display(df_sentences.head(10)) 


=== TÁCH CÂU THÀNH CÔNG (TỔNG CỘNG: 42 CÂU) ===



,STT,Nội dung câu
0,1,"Việt Nam thắng 4-2, xếp nhì bảng FIFA ASEAN Cup\nIndonesiaTiền đạo Nguyễn Đình Bắc ghi một bàn, kiến tạo hai bàn sau khi vào sân từ ghế dự bị, giúp Việt Nam thắng Pakistan 4-2 để giành suất tranh giải ba FIFA ASEAN Cup 2026."
1,2,"Việt Nam thắng 4-2, qua đó giành sáu điểm qua ba trận, xếp nhì bảng B. Thầy trò Kim Sang-sik sẽ đấu Malaysia ở trận tranh giải ba FIFA ASEAN Cup hôm 5/10."
2,3,Tiền đạo 19 tuổi ghi bàn đầu tiên cho đội tuyển.
3,4,"Từ đường chuyền của Đình Bắc, Minh Hoàng thoát xuống đánh bại thủ môn Pakistan trong pha đối mặt, nâng tỷ số lên 4-2."
4,5,Hiệp hai có 4 phút bù giờ.
5,6,"Ali Khan dứt điểm, bóng bất ngờ chạm chân Kaleem Ulah đổi hướng vào lưới."
6,7,"Patrik Lê Giang đã đổ người trước đó, nên không kịp phản ứng."
7,8,Phan Tuấn Tài chuyền vượt tuyến để Đình Bắc thoát xuống sau lưng hàng thủ Pakistan.
8,9,"Tiền đạo Việt Nam bình tĩnh dứt điểm trong thế đối mặt thủ môn, nâng cách biệt lên hai bàn."
9,10,Hai Long bứt tốc đón đường chuyền dài của Xuân Mạnh rồi căng ngang vượt qua thủ môn.


In [ ]:
# ==============================================================================
# BƯỚC 3: TÁCH TỪ (WORD SEGMENTATION)
# ==============================================================================
tokenized_sentences = []
raw_tokens = []

for s in sentences:
    s_clean = re.sub(r"\s+", " ", s).strip()
    words = vi_word_tokenize(s_clean)
    tokenized_sentences.append(words)
    
    for w in words:
        w_lower = w.lower().strip()
        if re.search(r"\w", w_lower):
            raw_tokens.append(w_lower)

print(f"=== TỔNG SỐ TOKEN TRƯỚC KHI LỌC STOPWORDS: {len(raw_tokens)} tokens ===")
print(f"=== KÍCH THƯỚC VOCABULARY GỐC (DUY NHẤT): {len(set(raw_tokens))} từ ===")


=== TỔNG SỐ TOKEN TRƯỚC KHI LỌC STOPWORDS: 616 tokens ===
=== KÍCH THƯỚC VOCABULARY GỐC (DUY NHẤT): 339 từ ===


In [ ]:
# ==============================================================================
# BƯỚC 4: LOẠI BỎ TỪ DỪNG (STOPWORDS REMOVAL)
# ==============================================================================

vietnamese_stopwords = {
    "và", "là", "của", "ở", "có", "một", "những", "các", "cho", "với",
    "đã", "đang", "sẽ", "được", "bị", "trong", "để", "khi", "nhưng", "từ",
    "ra", "vào", "lên", "xuống", "qua", "sau", "trước", "này", "đó", "thì",
    "mà", "cũng", "như", "thế", "vì", "do", "nên", "lại", "rồi", "quá",
    "rất", "nhiều", "ít", "hơn", "nhất", "về", "theo", "cùng", "nơi", "tại",
    "không", "chưa", "chẳng", "phải", "ngay", "cả", "bằng"
}


filtered_tokens = [t for t in raw_tokens if t not in vietnamese_stopwords]

filtered_sentences = []
for words in tokenized_sentences:
    clean_sent = [w for w in words if w.lower().strip() not in vietnamese_stopwords and re.search(r"\w", w)]
    filtered_sentences.append(clean_sent)

print("=== KẾT QUẢ SAU KHI LỌC STOPWORDS ===")
print(f"- Số lượng token ban đầu : {len(raw_tokens)}")
print(f"- Số lượng token sau lọc : {len(filtered_tokens)} (Giảm {len(raw_tokens) - len(filtered_tokens)} stopwords)")
print(f"- Kích thước Vocab sau lọc: {len(set(filtered_tokens))} từ duy nhất\n")

print("--- Ví dụ Câu 1 ---")
print("Tokens gốc     :", tokenized_sentences[0])
print("Tokens sau lọc :", filtered_sentences[0])


=== KẾT QUẢ SAU KHI LỌC STOPWORDS ===
- Số lượng token ban đầu : 616
- Số lượng token sau lọc : 469 (Giảm 147 stopwords)
- Kích thước Vocab sau lọc: 299 từ duy nhất

--- Ví dụ Câu 1 ---
Tokens gốc     : ['Việt Nam', 'thắng', '4-2', ',', 'xếp', 'nhì', 'bảng', 'FIFA', 'ASEAN Cup', 'IndonesiaTiền đạo', 'Nguyễn Đình Bắc', 'ghi', 'một', 'bàn', ',', 'kiến tạo', 'hai', 'bàn', 'sau', 'khi', 'vào', 'sân', 'từ', 'ghế', 'dự bị', ',', 'giúp', 'Việt Nam', 'thắng', 'Pakistan 4-2', 'để', 'giành suất', 'tranh', 'giải', 'ba', 'FIFA ASEAN', 'Cup', '2026', '.']
Tokens sau lọc : ['Việt Nam', 'thắng', '4-2', 'xếp', 'nhì', 'bảng', 'FIFA', 'ASEAN Cup', 'IndonesiaTiền đạo', 'Nguyễn Đình Bắc', 'ghi', 'bàn', 'kiến tạo', 'hai', 'bàn', 'sân', 'ghế', 'dự bị', 'giúp', 'Việt Nam', 'thắng', 'Pakistan 4-2', 'giành suất', 'tranh', 'giải', 'ba', 'FIFA ASEAN', 'Cup', '2026']


In [23]:
# ==============================================================================
# BƯỚC 5: TẠO VÀ SO SÁNH TẬP TỪ VỰNG (VOCABULARY) TRƯỚC VÀ SAU LỌC STOPWORDS
# ==============================================================================
raw_counts = Counter(raw_tokens)
filtered_counts = Counter(filtered_tokens)

df_raw_top = pd.DataFrame(raw_counts.most_common(10), columns=["Từ gốc", "Tần suất gốc"])

df_clean_top = pd.DataFrame(filtered_counts.most_common(10), columns=["Từ khóa chính (Clean)", "Tần suất sạch"])

df_comparison = pd.concat([df_raw_top, df_clean_top], axis=1)
print("=== BẢNG SO SÁNH TOP 10 TỪ VỰNG: TRƯỚC vs SAU KHI LỌC STOPWORDS ===")
display(df_comparison)

clean_vocab = sorted(list(set(filtered_tokens)))
df_clean_vocab = pd.DataFrame({
    "STT": list(range(1, len(clean_vocab) + 1)),
    "Từ vựng sau lọc (Clean Vocab)": clean_vocab,
    "Số lần xuất hiện": [filtered_counts[w] for w in clean_vocab]
}).sort_values(by="Số lần xuất hiện", ascending=False).reset_index(drop=True)
df_clean_vocab["STT"] = list(range(1, len(df_clean_vocab) + 1))

print(f"\n=== BẢNG TOÀN BỘ TẬP TỪ VỰNG ĐÃ LỌC STOPWORDS ({len(clean_vocab)} TỪ) ===")
display(df_clean_vocab.head(20)) # Hiển thị 20 từ vựng nổi bật nhất


=== BẢNG SO SÁNH TOP 10 TỪ VỰNG: TRƯỚC vs SAU KHI LỌC STOPWORDS ===


,Từ gốc,Tần suất gốc,Từ khóa chính (Clean),Tần suất sạch
0,việt nam,15,việt nam,15
1,hai,10,hai,10
2,trong,10,bóng,8
3,của,9,trận,7
4,sau,8,pakistan,7
5,bóng,8,dứt điểm,7
6,và,8,đường,6
7,trận,7,thủ môn,6
8,pakistan,7,bàn,5
9,dứt điểm,7,ba,5



=== BẢNG TOÀN BỘ TẬP TỪ VỰNG ĐÃ LỌC STOPWORDS (299 TỪ) ===


,STT,Từ vựng sau lọc (Clean Vocab),Số lần xuất hiện
0,1,việt nam,15
1,2,hai,10
2,3,bóng,8
3,4,pakistan,7
4,5,trận,7
5,6,dứt điểm,7
6,7,đường,6
7,8,thủ môn,6
8,9,kim sang-sik,5
9,10,bàn,5
